# Part A: Deeper CNN with BatchNormalization and Enhanced Regularization
## Standard TensorFlow Practice - Advanced Architecture

This notebook demonstrates:
- Deeper CNN architecture with BatchNormalization
- Layer normalization techniques
- Advanced regularization (L1/L2, dropout)
- Performance comparison with baseline
- tf.data optimization for large models

## 1. Install and Import Libraries

In [ ]:
# Install required packages
!pip install tensorflow>=2.13 tensorflow-gpu torch torchvision scikit-learn -q

import tensorflow as tf
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)

## 2. GPU Configuration and Detection

In [ ]:
# Check TensorFlow GPU availability
print("\n=== TensorFlow GPU Configuration ===")
tf_gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow GPU Available: {len(tf_gpus) > 0}")
if tf_gpus:
    for gpu in tf_gpus:
        print(f"  - {gpu}")

# Configure TensorFlow to use GPU memory growth
for gpu in tf_gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Check PyTorch GPU availability
print("\n=== PyTorch GPU Configuration ===")
print(f"PyTorch GPU Available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA Version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

# Set default device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Default PyTorch Device: {device}")

## 3. Load and Prepare Dataset

In [ ]:
# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten labels
y_train = y_train.flatten()
y_test = y_test.flatten()

print(f"\nDataset shapes:")
print(f"  Training images: {x_train.shape}")
print(f"  Training labels: {y_train.shape}")
print(f"  Test images: {x_test.shape}")
print(f"  Test labels: {y_test.shape}")

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(class_names)
print(f"\nNumber of classes: {num_classes}")

## 4. Create Optimized tf.data Pipeline

In [ ]:
# Hyperparameters
BATCH_SIZE = 128
IMG_SIZE = 32

def create_dataset(x, y, batch_size, augment=False):
    """
    Create an optimized tf.data.Dataset pipeline with advanced augmentation.
    
    Args:
        x: Input images
        y: Labels
        batch_size: Batch size
        augment: Whether to apply data augmentation
    
    Returns:
        Optimized tf.data.Dataset
    """
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    
    if augment:
        # Advanced data augmentation for training
        def augment_fn(image, label):
            # Geometric augmentations
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_flip_up_down(image)
            
            # Rotation via padding and cropping
            image = tf.image.pad_to_bounding_box(image, 4, 4, 40, 40)
            image = tf.image.random_crop(image, [32, 32, 3])
            
            # Color augmentations
            image = tf.image.random_brightness(image, 0.2)
            image = tf.image.random_contrast(image, 0.8, 1.2)
            image = tf.image.random_saturation(image, 0.8, 1.2)
            
            # Clip to valid range
            image = tf.clip_by_value(image, 0.0, 1.0)
            return image, label
        
        dataset = dataset.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Batch and prefetch for GPU optimization
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Create training and validation datasets
num_train = int(0.9 * len(x_train))
indices = np.random.permutation(len(x_train))

x_train_split = x_train[indices[:num_train]]
y_train_split = y_train[indices[:num_train]]

x_val = x_train[indices[num_train:]]
y_val = y_train[indices[num_train:]]

# Create tf.data datasets
train_dataset = create_dataset(x_train_split, y_train_split, BATCH_SIZE, augment=True)
val_dataset = create_dataset(x_val, y_val, BATCH_SIZE, augment=False)
test_dataset = create_dataset(x_test, y_test, BATCH_SIZE, augment=False)

print(f"Training dataset batches: {len(train_dataset)}")
print(f"Validation dataset batches: {len(val_dataset)}")
print(f"Test dataset batches: {len(test_dataset)}")

## 5. Build Deeper CNN Model with BatchNormalization

In [ ]:
def create_deeper_model(input_shape=(32, 32, 3), num_classes=10):
    """
    Create a deeper CNN model with BatchNormalization and enhanced regularization.
    
    Architecture:
    - 5 convolutional blocks
    - BatchNormalization after each conv layer
    - L2 regularization
    - Progressive dropout
    
    Args:
        input_shape: Shape of input images
        num_classes: Number of output classes
    
    Returns:
        Compiled Keras model
    """
    l2_reg = tf.keras.regularizers.l2(0.0001)
    
    model = tf.keras.Sequential([
        # Input layer
        tf.keras.layers.Input(shape=input_shape),
        
        # Block 1: 32 filters
        tf.keras.layers.Conv2D(32, (3, 3), padding='same', kernel_regularizer=l2_reg, name='conv1_1'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        
        tf.keras.layers.Conv2D(32, (3, 3), padding='same', kernel_regularizer=l2_reg, name='conv1_2'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        
        tf.keras.layers.MaxPooling2D((2, 2), name='pool1'),
        tf.keras.layers.Dropout(0.25),
        
        # Block 2: 64 filters
        tf.keras.layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=l2_reg, name='conv2_1'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        
        tf.keras.layers.Conv2D(64, (3, 3), padding='same', kernel_regularizer=l2_reg, name='conv2_2'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        
        tf.keras.layers.MaxPooling2D((2, 2), name='pool2'),
        tf.keras.layers.Dropout(0.25),
        
        # Block 3: 128 filters
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=l2_reg, name='conv3_1'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', kernel_regularizer=l2_reg, name='conv3_2'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        
        tf.keras.layers.MaxPooling2D((2, 2), name='pool3'),
        tf.keras.layers.Dropout(0.3),
        
        # Block 4: 256 filters
        tf.keras.layers.Conv2D(256, (3, 3), padding='same', kernel_regularizer=l2_reg, name='conv4_1'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        
        tf.keras.layers.Conv2D(256, (3, 3), padding='same', kernel_regularizer=l2_reg, name='conv4_2'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        
        tf.keras.layers.Dropout(0.3),
        
        # Global Average Pooling
        tf.keras.layers.GlobalAveragePooling2D(),
        
        # Dense layers
        tf.keras.layers.Dense(512, kernel_regularizer=l2_reg, name='dense1'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(256, kernel_regularizer=l2_reg, name='dense2'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Dropout(0.5),
        
        # Output layer
        tf.keras.layers.Dense(num_classes, activation='softmax', name='output')
    ])
    
    return model

# Create model
model = create_deeper_model()

# Display model architecture
print("\n=== Deeper CNN Model Architecture ===")
model.summary()

## 6. Model Comparison Metrics

## 7. Compile Model with Advanced Optimization

## 8. Define Advanced Callbacks

## 9. Train Deeper Model (High-Level API)